# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [18]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# install openwakeword (full installation to support training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 184 (delta 70), reused 53 (delta 53), pack-reused 98 (from 1)
Receiving objects: 100% (184/184), 1.04 MiB | 7.67 MiB/s, done.
Resolving deltas: 100% (93/93), done.
--2026-08-02 11:46:19--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-08-02T12%3A40%3A49Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-4

In [111]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

# Cài đặt thư viện onnx cần thiết cho việc chuyển đổi model
!pip install onnx
!pip install onnx_tf

  Using cached onnx_tf-1.10.0-py3-none-any.whl.metadata (510 bytes)
INFO: pip is looking at multiple versions of onnx-tf to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.6/186.6 kB 4.9 MB/s eta 0:00:00


In [101]:
# Cell này đã bị loại bỏ vì `!pip install onnx` đã được chuyển vào ô Imports (d4c1056e).
# This cell has been removed as `!pip install onnx` has been moved to the Imports cell (d4c1056e).

# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [103]:
# Download room impulse responses collected by MIT
import os
import datasets
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm

output_dir = "/content/mit_rirs"
os.makedirs(output_dir, exist_ok=True)

print("--- Đang tải dữ liệu MIT RIRs qua HuggingFace datasets... ---")
try:
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

    print("--- Đang lưu các clip RIR vào tệp WAV 16-bit PCM... ---")
    num_downloaded = 0
    # The dataset contains 270 files. Use a counter to ensure we don't go past expected items in streaming mode.
    for i, row in tqdm(enumerate(rir_dataset), total=270):
        if num_downloaded >= 270:
            break
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
        num_downloaded += 1
    print(f"Đã tải xuống và lưu {num_downloaded} tệp RIR vào {output_dir}")

except ConnectionError as e:
    print(f"LỖI KẾT NỐI khi tải MIT RIRs từ HuggingFace: {e}")
    print("Vấn đề này thường là tạm thời. Vui lòng chạy lại ô này để thử lại.")
    print("Nếu lỗi vẫn tiếp diễn, có thể có vấn đề với kết nối mạng hoặc máy chủ HuggingFace.")
    raise # Re-raise the error to stop execution and notify the user
except Exception as e:
    print(f"Đã xảy ra lỗi không mong muốn khi tải MIT RIRs: {e}")
    raise


print(f"Thư mục {output_dir} tồn tại: {os.path.exists(output_dir)}")
if os.path.exists(output_dir):
    num_files = len([f for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))])
    print(f"Số lượng file trong {output_dir}: {num_files}")
    if num_files == 0:
        print(f"CẢNH BÁO: Thư mục {output_dir} trống rỗng sau khi giải nén/tải xuống.")
else:
    print(f"LỖI: Thư mục {output_dir} không tồn tại sau khi xử lý.")


--- Đang tải dữ liệu MIT RIRs qua HuggingFace datasets... ---


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

--- Đang lưu các clip RIR vào tệp WAV 16-bit PCM... ---


100%|██████████| 270/270 [00:42<00:00,  6.37it/s]

Đã tải xuống và lưu 270 tệp RIR vào /content/mit_rirs
Thư mục /content/mit_rirs tồn tại: True
Số lượng file trong /content/mit_rirs: 270


In [95]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

output_dir_audioset_16k = "/content/audioset_16k"
os.makedirs(output_dir_audioset_16k, exist_ok=True)

print("Downloading a subset of AudioSet using HuggingFace datasets...")
# Using datasets.load_dataset directly for a small number of samples
# Note: Streaming=True might be slow for large datasets, but for a few samples it's fine.
# For larger scale, consider downloading the full dataset offline.
audioset_streaming_dataset = datasets.load_dataset("agkphysics/AudioSet", split="train", streaming=True)
audioset_streaming_dataset = audioset_streaming_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

n_audioset_samples = 100 # Adjust as needed for a quick example
for i, row in tqdm(enumerate(audioset_streaming_dataset), total=n_audioset_samples):
    if i >= n_audioset_samples:
        break
    # Use a generic name for each sample as the original path might be complex
    name = f"audioset_sample_{i:04d}.wav"
    scipy.io.wavfile.write(os.path.join(output_dir_audioset_16k, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print(f"Downloaded {min(i+1, n_audioset_samples)} AudioSet samples to {output_dir_audioset_16k}")

# Free Music Archive dataset (https://github.com/mdeff/fma)
output_dir_fma = "/content/fma"
os.makedirs(output_dir_fma, exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # use only 1 hour of clips for this example notebook, recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # this works because the FMA dataset is all 30 second clips
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir_fma, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/870 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/870 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

TypeError: must be called with a dataclass type or instance

In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [28]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)
config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [110]:
# Modify values in the config and save a new version

config["target_phrase"] = ["hey lucy"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

config["background_paths"] = ['./audioset_16k', './fma']  # multiple background datasets are supported
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

# Explicitly set the TTS model path
config["tts_model_path"] = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

In [88]:
# Cell này đã được loại bỏ vì các hàm tải xuống dữ liệu khác đã được sửa đổi và ưu tiên sử dụng.
# This cell has been removed as other data download functions have been modified and prioritized.

In [89]:
# Cell này đã được loại bỏ vì các hàm tải xuống dữ liệu khác đã được sửa đổi và ưu tiên sử dụng.
# This cell has been removed as other data download functions have been modified and prioritized.

In [62]:
# Chuyển đổi toàn bộ đường dẫn tương đối trong file config sang tuyệt đối
config_path = "my_model.yaml"

with open(config_path, "r") as f:
    content = f.read()

# Thay thế các đường dẫn tương đối thành tuyệt đối
content = content.replace("- ./mit_rirs", "- /content/mit_rirs")
content = content.replace("- ./audioset_16k", "- /content/audioset_16k")
content = content.replace("- ./fma", "- /content/fma")
content = content.replace("output_dir: ./my_custom_model", "output_dir: /content/my_custom_model")
content = content.replace("piper_sample_generator_path: ./piper-sample-generator", "piper_sample_generator_path: /content/piper-sample-generator")

with open(config_path, "w") as f:
    f.write(content)

print("Đã cập nhật cấu hình sang đường dẫn tuyệt đối thành công!")


Đã cập nhật cấu hình sang đường dẫn tuyệt đối thành công!


In [58]:
!pip install espeak-phonemizer


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 46.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for espeak-phonemizer: filename=espeak_phonemizer-1.3.1-py3-none-any.whl size=19792 sha256=acde8739a38a5ffe71d431a9f2e38b5d5bc8b3a004c7028722877436efb15f51
  Stored in directory: /root/.cache/pip/wheels/4e/14/2f/a1bc1cb50727d00abfde50744386e5f0ea2b5e27357c354c7d
Successfully built espeak-phonemizer


In [109]:
!rm -rf /content/piper-sample-generator
!git clone https://github.com/dscripka/piper-sample-generator /content/piper-sample-generator
!wget -O /content/piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

Cloning into '/content/piper-sample-generator'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 75 (delta 21), reused 18 (delta 18), pack-reused 40 (from 1)
Receiving objects: 100% (75/75), 1.01 MiB | 4.09 MiB/s, done.
Resolving deltas: 100% (22/22), done.
--2026-08-02 13:32:36--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-08-02T14%3A30%3A26Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-9

In [90]:
# Cell này đã được loại bỏ vì các hàm tải xuống dữ liệu khác đã được sửa đổi và ưu tiên sử dụng.
# This cell has been removed as other data download functions have been modified and prioritized.

In [74]:
import os
print("Danh sách file hiện tại trong /content:")
print(os.listdir("/content"))


Danh sách file hiện tại trong /content:
['.config', 'fma', 'fma.tar.gz', 'mit_rirs.zip', 'audioset_16k.tar.gz', 'audioset_16k', 'my_custom_model', 'piper-sample-generator', 'my_model.yaml', 'openwakeword', 'sample_data']


In [59]:
import os

target_path = "/content/openwakeword/openwakeword/train.py"
if os.path.exists(target_path):
    print(f"--- 10 dòng đầu của file {target_path}: ---")
    with open(target_path, "r") as f:
        lines = f.readlines()
        for i in range(min(10, len(lines))):
            print(f"{i+1}: {lines[i]}", end="")
else:
    print("File không tồn tại!")


--- 10 dòng đầu của file /content/openwakeword/openwakeword/train.py: ---
1: import sys
2: sys.path.append('/content/piper-sample-generator')
3: sys.path.append('/content/piper-sample-generator/piper_sample_generator')
4: import torch
5: from torch import optim, nn
6: import torchinfo
7: import torchmetrics
8: import copy
9: import os
10: import sys


In [132]:
import os
import re # Import regex module

# 1. KHÔNG KHÔI PHỤC file train.py sạch gốc bằng Git để giữ lại các bản vá
print("1. Không khôi phục file train.py từ Git để giữ các bản vá.")

# 2. Khai báo các đường dẫn có thể có
possible_paths = [
    "/content/openwakeword/openwakeword/train.py",
    "/content/openWakeWord/openwakeword/train.py"
]

target_file = None
for p in possible_paths:
    if os.path.exists(p):
        target_file = p
        break

if target_file:
    print(f"2. Đã tìm thấy file train.py tại: {target_file}")
    with open(target_file, "r") as f:
        code_content = f.read()

    # Clean up all previous sys.path.append for piper-sample-generator and rebuild sys imports cleanly
    lines = code_content.splitlines()
    new_lines_filtered = []
    sys_import_found = False

    for line in lines:
        # Remove sys.path.append lines related to piper-sample-generator
        if "sys.path.append('/content/piper-sample-generator')" in line or \
           "sys.path.append('/content/piper-sample-generator/piper_sample_generator')" in line:
            continue
        # Keep only the first 'import sys'
        if "import sys" in line and not sys_import_found:
            sys_import_found = True
            new_lines_filtered.append(line)
            continue
        elif "import sys" in line and sys_import_found:
            continue # Skip subsequent 'import sys'

        new_lines_filtered.append(line)

    code_content_cleaned = "\n".join(new_lines_filtered)

    # Re-insert sys imports and path appends at the top
    header_to_add = []
    if not sys_import_found:
        header_to_add.append("import sys")
    header_to_add.append("sys.path.append('/content/piper-sample-generator')")
    header_to_add.append("sys.path.append('/content/piper-sample-generator/piper_sample_generator')")

    # Prepend the header to the cleaned code_content
    final_code = "\n".join(header_to_add) + "\n" + code_content_cleaned.strip()

    # Patch for generate_samples call to explicitly pass model_path from config
    temp_lines = final_code.splitlines()
    new_temp_lines = []
    modified_model_path_count = 0
    in_generate_samples_block = False
    model_path_handled_in_current_block = False

    for i, line in enumerate(temp_lines):
        if re.search(r"generate_samples\(", line): # Found the start of a generate_samples call
            in_generate_samples_block = True
            model_path_handled_in_current_block = False # Reset for this block
            new_temp_lines.append(line)
        elif in_generate_samples_block:
            if re.search(r"^\s*model_path=", line): # Check if the line contains 'model_path='
                # Found an existing model_path argument, replace it
                new_temp_lines.append("            model_path=config['tts_model_path'], # Updated by Colab agent")
                modified_model_path_count += 1
                model_path_handled_in_current_block = True
            elif re.search(r"^\s*\)", line): # Found the closing parenthesis of the call
                if not model_path_handled_in_current_block:
                    # Insert model_path argument before the closing parenthesis
                    new_temp_lines.append("            model_path=config['tts_model_path'], # Added by Colab agent")
                    modified_model_path_count += 1
                new_temp_lines.append(line)
                in_generate_samples_block = False
            else:
                new_temp_lines.append(line) # Regular line within the call
        else:
            new_temp_lines.append(line) # Not in generate_samples call

    if modified_model_path_count == 0:
        print("   -> KHÔNG tìm thấy hoặc không thể vá lỗi tham số 'model_path' trong generate_samples call.")
    else:
        print(f"   -> Đã vá lỗi {modified_model_path_count} tham số 'model_path' trong generate_samples call.")

    final_code = "\n".join(new_temp_lines)

    with open(target_file, "w") as f:
        f.write(final_code)

    print("   -> Đã vá lỗi đường dẫn import và tham số model thành công!")
else:
    print("2. KHÔNG TÌM THẤY file train.py!")

# 3. Vá lỗi torchaudio (existing code, no changes)
filepath = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"
if os.path.exists(filepath):
    with open(filepath, "r") as f:
        io_code = f.read()
    # Ensure the replacement only happens if the original line exists
    if 'torchaudio.set_audio_backend("soundfile")' in io_code:
        io_code = io_code.replace('torchaudio.set_audio_backend("soundfile")', 'pass')
        with open(filepath, "w") as f:
            f.write(io_code)
        print("3. Đã sửa lỗi torchaudio thành công!")
    else:
        print("3. Dòng lỗi torchaudio đã được sửa hoặc không tồn tại.")
else:
    print(f"3. File torchaudio không tồn tại tại {filepath}!")

1. Không khôi phục file train.py từ Git để giữ các bản vá.
2. Đã tìm thấy file train.py tại: /content/openwakeword/openwakeword/train.py
   -> Đã vá lỗi 3 tham số 'model_path' trong generate_samples call.
   -> Đã vá lỗi đường dẫn import và tham số model thành công!
3. Dòng lỗi torchaudio đã được sửa hoặc không tồn tại.


In [36]:
import os
print("Thư mục hiện tại (Working Directory):", os.getcwd())
print("\nDanh sách các file/thư mục ở đây:")
print(os.listdir(os.getcwd()))


Thư mục hiện tại (Working Directory): /content

Danh sách các file/thư mục ở đây:
['.config', 'piper-sample-generator', 'my_model.yaml', 'openwakeword', 'sample_data']


In [35]:
import os

print("Đang quét tìm file train.py...")
found = False
for root, dirs, files in os.walk("/content"):
    for file in files:
        if "train.py" in file:
            print(f"-> Tìm thấy: {os.path.join(root, file)}")
            found = True

if not found:
    print("Không tìm thấy bất kỳ file nào tên là train.py trong thư mục /content!")


Đang quét tìm file train.py...
-> Tìm thấy: /content/openwakeword/openwakeword/train.py


In [131]:
<!-- This cell has been removed to prevent conflicts with the PYTHONPATH environment variable. -->

SyntaxError: invalid syntax (1198185476.py, line 1)

In [39]:
# Vá lỗi torchaudio.set_audio_backend trong thư viện torch_audiomentations
filepath = "/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py"

with open(filepath, "r") as f:
    code = f.read()

# Thay thế dòng bị lỗi thành pass (bỏ qua)
code = code.replace('torchaudio.set_audio_backend("soundfile")', 'pass')

with open(filepath, "w") as f:
    f.write(code)

print("Đã sửa lỗi torchaudio thành công!")


Đã sửa lỗi torchaudio thành công!


# Cài đặt thư viện onnx cần thiết cho việc chuyển đổi model
!pip install onnx


In [118]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!python openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:generate_samples:Loading /content/piper-sample-generator/models/en-us-libritts-high.pt
Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 682, in <module>
    generate_samples(
  File "/content/piper-sample-generator/generate_samples.py", line 74, in generate_samples
    model = torch.load(model_path)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 776, in __init__
 

In [119]:
# Step 2: Augment the generated clips

!python openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 757, in <module>
    sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "numpy/random/mtrand.pyx", line 798, in numpy.random.mtrand.RandomState.randint
  File "numpy/random/_bounded_integers.pyx", line 1334, in numpy.random._bounded_integers._rand_int64
ValueError: high <= 0


In [120]:
# Step 3: Train model

!python openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 757, in <module>
    sr, dat = scipy.io.wavfile.read(positive_clips[np.random.randint(0, len(positive_clips))])
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "numpy/random/mtrand.pyx", line 798, in numpy.random.mtrand.RandomState.randint
  File "numpy/random/_bounded_integers.pyx", line 1334, in numpy.random._bounded_integers._rand_int64
ValueError: high <= 0


In [133]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# --- TFLite conversion is skipped due to dependency issues with onnx_tf and tensorflow-addons. ---
# The primary training script (train.py) might still attempt the conversion.

# The content of this cell has been converted to markdown as it's an optional step
# that was causing issues and is currently being skipped.



After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!